Cell 1 – Install / Import

In [13]:
import pandas as pd
import numpy as np
import os
from datetime import datetime

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

Cell 2 – Prepare Data

In [14]:
DATA_PATH = "../data/WA_Fn-UseC_-Telco-Customer-Churn.csv"
MODEL_DIR = "../backend/models/telco/"
MODEL_NAME = "keras_model.h5"
FEATURES_NAME = "feature_names_keras.json"

os.makedirs(MODEL_DIR, exist_ok=True)

Cell 3 – Build & Train Model

In [15]:
df = pd.read_csv(DATA_PATH)
print(f"✅ Loaded dataset with shape: {df.shape}")

✅ Loaded dataset with shape: (7043, 21)


Cell 4 – Export .h5 Model

In [16]:
# Fix TotalCharges
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df = df.dropna()

# Encode target
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

# Drop ID
df = df.drop(columns=["customerID"])

# Split features / label
y = df["Churn"]
X = df.drop(columns=["Churn"])

# One-hot encode categoricals
X = pd.get_dummies(X)

# Scale numeric features
num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
scaler = StandardScaler()
X[num_cols] = scaler.fit_transform(X[num_cols])

In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"🧪 Training samples: {X_train.shape[0]}")
print(f"🧪 Testing samples:  {X_test.shape[0]}")

🧪 Training samples: 5625
🧪 Testing samples:  1407


In [18]:
model = Sequential([
    Dense(128, activation="relu", input_shape=(X_train.shape[1],)),
    Dropout(0.3),
    Dense(64, activation="relu"),
    Dropout(0.3),
    Dense(1, activation="sigmoid")  # Binary classification
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

/Users/darrylcarp/Coding/VScodeProjects/INFERSTREAM/.venv64/lib/python3.13/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [19]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.7727 - loss: 0.4714 - val_accuracy: 0.8044 - val_loss: 0.4062
Epoch 2/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 697us/step - accuracy: 0.8004 - loss: 0.4323 - val_accuracy: 0.8036 - val_loss: 0.4044
Epoch 3/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 725us/step - accuracy: 0.7996 - loss: 0.4260 - val_accuracy: 0.7982 - val_loss: 0.4207
Epoch 4/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 670us/step - accuracy: 0.8020 - loss: 0.4251 - val_accuracy: 0.8027 - val_loss: 0.4091
Epoch 5/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 647us/step - accuracy: 0.8044 - loss: 0.4187 - val_accuracy: 0.8089 - val_loss: 0.4081
Epoch 6/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 651us/step - accuracy: 0.8053 - loss: 0.4192 - val_accuracy: 0.7956 - val_loss: 0.4074
Epoch 7/100
141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 660us/step - accuracy: 0.8060 - loss: 0.4137 - val_accuracy: 0.8000 - val_loss: 0.4120


In [20]:
preds = (model.predict(X_test) > 0.5).astype(int)

print("📊 Classification Report:")
print(classification_report(y_test, preds, digits=4))

print("🧩 Confusion Matrix:")
print(confusion_matrix(y_test, preds))

44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 623us/step
📊 Classification Report:
              precision    recall  f1-score   support

           0     0.8527    0.8800    0.8661      1033
           1     0.6364    0.5802    0.6070       374

    accuracy                         0.8003      1407
   macro avg     0.7445    0.7301    0.7366      1407
weighted avg     0.7952    0.8003    0.7972      1407

🧩 Confusion Matrix:
[[909 124]
 [157 217]]


In [21]:
model.save(os.path.join(MODEL_DIR, MODEL_NAME))

X.columns.to_series().to_json(
    os.path.join(MODEL_DIR, FEATURES_NAME),
    indent=2
)

print(f"✅ Model saved to: {MODEL_DIR}{MODEL_NAME}")
print(f"🧠 Feature list saved to: {MODEL_DIR}{FEATURES_NAME}")
print("🏁 Done at", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

✅ Model saved to: ../backend/models/telco/keras_model.h5
🧠 Feature list saved to: ../backend/models/telco/feature_names_keras.json
🏁 Done at 2026-01-24 11:33:04
